# 기사 밀어내기(Churnalism) 검정 노트북
### 이슈 발생 시 "유사 기사 대량 생산"을 두 지표로 통계 검정

> **핵심 아이디어** — 이슈가 터지면 거의 똑같은 기사를 공장처럼 마구 찍어낸다.
> 이를 두 지표로 잡는다:
> 1. **기사 수 N_t** — 평소보다 유의하게 늘었는가? → **음이항 회귀**
> 2. **평균 코사인 유사도 S_t** — 평소보다 유의하게 닮았는가? → **Permutation test + 무작위 배정 영모형**
>
> 두 조건이 **모두** 충족될 때 churnalism으로 판정한다.
> ("풀리는지(평균회귀)"는 보지 않음 — 큰 클러스터(스노우볼)의 발생 자체가 관심.)

---
**작성**: sdkparkforbi · Colab / 로컬 공용 · 한국어 임베딩 기반


## 0. 환경 설정
한국어 폰트는 `koreanize_matplotlib` 사용(세션 재시작 불필요).

In [ ]:
# Colab/로컬 공용 설치 (필요한 것만)
import importlib, subprocess, sys

def ensure(pkg, import_name=None):
    name = import_name or pkg
    try:
        importlib.import_module(name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg, imp in [
    ("sentence-transformers", "sentence_transformers"),
    ("statsmodels", "statsmodels"),
    ("koreanize-matplotlib", "koreanize_matplotlib"),
    ("scikit-learn", "sklearn"),
]:
    ensure(pkg, imp)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib  # 한글 폰트 자동 적용 (apt nanum 설치 X)

np.random.seed(42)
print("준비 완료 ✅")


## 1. 설정 (여기만 바꾸면 됩니다)

- `TIME_UNIT` : 시간 해상도 (`'D'` 일 단위 / `'H'` 시간 단위)
- `EVENT_TIME` : 이슈 발생 시점 (이 시점 이후가 '사후')
- `PRE_DAYS`, `POST_DAYS` : 사전·사후 기간(일)
- `EMB_MODEL` : 한국어 문장 임베딩 모델


In [ ]:
# ====== 사용자 설정 ======
TIME_UNIT  = "D"                      # 'D'=일별, 'H'=시간별
EVENT_TIME = "2026-01-10 00:00:00"    # 이슈 발생 시점
PRE_DAYS   = 7                        # 사전 기간(기준선 추정용)
POST_DAYS  = 7                        # 사후 기간
EMB_MODEL  = "jhgan/ko-sroberta-multitask"   # 한국어 SBERT (대안: snunlp/KR-SBERT-V40K-klueNLI-augSTS)
N_PERM     = 5000                     # permutation 반복 수
N_NULLSIM  = 1000                     # 무작위 배정 영모형 반복 수
# ========================
EVENT_TIME = pd.Timestamp(EVENT_TIME)
print(f"시간단위={TIME_UNIT} | 이슈시점={EVENT_TIME} | 임베딩={EMB_MODEL}")


## 2. 데이터 불러오기

기대 형식: 컬럼 `published_at`(날짜/시간), `text`(기사 본문 또는 제목+본문).

아래는 **예시 합성 데이터**입니다. 실제 데이터가 있으면 이 셀을 교체하세요:
```python
df = pd.read_csv("your_news.csv", parse_dates=["published_at"])
df = df[["published_at", "text"]].dropna()
```


In [ ]:
# ===== 실제 데이터가 있으면 이 블록을 교체 =====
# df = pd.read_csv("your_news.csv", parse_dates=["published_at"])
# df = df[["published_at","text"]].dropna()

# ----- (데모용) 합성 데이터 생성 -----
def make_demo():
    rng = np.random.default_rng(0)
    rows = []
    base = EVENT_TIME - pd.Timedelta(days=PRE_DAYS)
    # 사전: 다양한 주제, 적은 양
    topics_pre = ["경제 물가 상승", "날씨 한파 주의보", "야구 경기 결과",
                  "영화 개봉 소식", "건강 운동 습관", "여행 명소 추천"]
    for d in range(PRE_DAYS):
        day = base + pd.Timedelta(days=d)
        n = rng.integers(8, 16)                       # 평소: 하루 8~15건
        for _ in range(n):
            t = rng.choice(topics_pre)
            hour = int(rng.integers(6, 23))
            rows.append((day + pd.Timedelta(hours=hour),
                         t + " " + " ".join(rng.choice(["분석","전망","현장","정리","속보"], 2))))
    # 사후: 한 이슈로 유사 기사 폭증
    issue = "A기업 신약 임상 3상 성공 발표 주가 급등"
    side  = ["경제 물가 상승", "날씨 한파 주의보", "영화 개봉 소식"]
    for d in range(POST_DAYS):
        day = EVENT_TIME + pd.Timedelta(days=d)
        boost = max(0, 60 - d*8)                       # 발생 직후 폭증 후 점차 감소
        n_issue = rng.integers(boost, boost+20) if boost>0 else rng.integers(2,6)
        n_side  = rng.integers(6, 12)
        for _ in range(n_issue):
            hour = int(rng.integers(0, 24))
            tail = " ".join(rng.choice(["발표","기대","호재","분석","전망","수혜주"], 2))
            rows.append((day + pd.Timedelta(hours=hour), issue + " " + tail))
        for _ in range(n_side):
            t = rng.choice(side)
            hour = int(rng.integers(6, 23))
            rows.append((day + pd.Timedelta(hours=hour), t + " 소식"))
    return pd.DataFrame(rows, columns=["published_at","text"])

df = make_demo()
df = df.sort_values("published_at").reset_index(drop=True)
print(f"총 기사 수: {len(df)}")
df.head()


## 3. 임베딩
각 기사를 문장 임베딩(숫자 벡터)으로 바꾸고 **단위벡터로 정규화**합니다. (정규화하면 평균 유사도를 빠르게 계산 가능)

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(EMB_MODEL)
emb = model.encode(df["text"].tolist(), batch_size=64,
                   show_progress_bar=True, normalize_embeddings=True)
emb = np.asarray(emb, dtype=np.float32)
print("임베딩 shape:", emb.shape)


## 4. 시간대별 지표 계산: N_t, S_t

**평균 코사인 유사도(빠른 공식)** — 한 구간에 단위벡터 $v_1..v_n$ 이 있을 때 모든 쌍의 평균 코사인은
$$ S = \frac{\lVert \bar v \rVert^2 \, n - 1}{\,n-1\,}, \quad \bar v = \frac1n\sum v_i $$
즉 $S = \dfrac{n\,\lVert\bar v\rVert^2 - 1}{n-1}$ — 모든 쌍을 일일이 돌지 않아도 됩니다.


In [ ]:
def mean_pairwise_cosine(vectors):
    """단위벡터 배열 -> 모든 쌍 평균 코사인 (빠른 공식). n<2면 NaN."""
    n = len(vectors)
    if n < 2:
        return np.nan
    mean_vec = vectors.mean(axis=0)
    return (n * float(mean_vec @ mean_vec) - 1.0) / (n - 1)

# 시간 버킷 부여
df["bucket"] = df["published_at"].dt.floor(TIME_UNIT)

records = []
idx_by_bucket = {}
for b, g in df.groupby("bucket"):
    idx = g.index.values
    idx_by_bucket[b] = idx
    records.append({
        "bucket": b,
        "N": len(idx),
        "S": mean_pairwise_cosine(emb[idx]),
        "period": "pre" if b < EVENT_TIME else "post",
    })

ts = pd.DataFrame(records).sort_values("bucket").reset_index(drop=True)
print(ts.to_string(index=False))


## 5. 시각화 — 두 지표 + 평소 줄(기준선)
사전 구간의 평균±표준편차로 기준선을 긋고, 사후에 어디서 넘는지 봅니다.

In [ ]:
pre = ts[ts.period=="pre"]
muN, sdN = pre["N"].mean(), pre["N"].std(ddof=1)
muS, sdS = pre["S"].mean(), pre["S"].std(ddof=1)

fig, ax = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

# (1) 기사 수
colors = ["#5BA8FF" if p=="pre" else "#FF7A6B" for p in ts.period]
ax[0].bar(range(len(ts)), ts["N"], color=colors)
ax[0].axhline(muN, ls="--", color="#E0954F", lw=2, label=f"평소 평균 {muN:.1f}")
ax[0].axhline(muN+3*sdN, ls=":", color="#C23A2B", lw=2, label="평소 +3σ (관리한계)")
ax[0].axvline(len(pre)-0.5, color="gray", lw=1)
ax[0].set_ylabel("기사 수 N")
ax[0].set_title("① 기사 수 — 평소 줄(기준선) 대비")
ax[0].legend(fontsize=9)

# (2) 평균 유사도
ax[1].bar(range(len(ts)), ts["S"], color=colors)
ax[1].axhline(muS, ls="--", color="#E0954F", lw=2, label=f"평소 평균 {muS:.3f}")
ax[1].axhline(muS+3*sdS, ls=":", color="#C23A2B", lw=2, label="평소 +3σ (관리한계)")
ax[1].axvline(len(pre)-0.5, color="gray", lw=1)
ax[1].set_ylabel("평균 코사인 유사도 S")
ax[1].set_title("② 평균 유사도 — 평소 줄(기준선) 대비")
ax[1].set_xticks(range(len(ts)))
ax[1].set_xticklabels([b.strftime("%m-%d %Hh" if TIME_UNIT=="H" else "%m-%d") for b in ts.bucket],
                      rotation=45, ha="right", fontsize=8)
ax[1].legend(fontsize=9)

plt.tight_layout(); plt.show()
print("회색 세로선 = 이슈 발생 시점 (왼쪽=사전, 오른쪽=사후)")


## 6. 검정 ① 기사 수 — 음이항 회귀

$$\log(\mu_t) = \beta_0 + \beta_1\,\text{post}_t + \text{요일더미} + (\text{시간대더미})$$

- **H₀:** $\beta_1 = 0$ (사후 기사 수 = 사전) &nbsp; **H₁:** $\beta_1 > 0$ (단측)
- 뉴스 기사 수는 **과대산포**가 커서 Poisson 대신 **음이항(NB)** 사용.
- $\exp(\beta_1)$ = "사후에 몇 배 늘었나".


In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

d = ts.copy()
d["post"] = (d.period=="post").astype(int)
d["dow"]  = d["bucket"].dt.dayofweek.astype("category")     # 요일 통제
formula = "N ~ post + C(dow)"
if TIME_UNIT == "H":
    d["hour"] = d["bucket"].dt.hour.astype("category")
    formula += " + C(hour)"                                 # 시간대 통제

# 과대산포 확인 (분산/평균)
overdisp = d["N"].var(ddof=1) / d["N"].mean()
print(f"분산/평균 = {overdisp:.2f}  (>1 이면 과대산포 → 음이항 적합)\n")

# 음이항 alpha 추정(보조 OLS) 후 NB 적합
poi = smf.glm(formula, data=d, family=sm.families.Poisson()).fit()
mu  = poi.mu
aux = ((d["N"]-mu)**2 - d["N"]) / mu
alpha = max(sm.OLS(aux, mu).fit().params.iloc[0], 1e-6)

nb = smf.glm(formula, data=d, family=sm.families.NegativeBinomial(alpha=alpha)).fit()
print(nb.summary())

b1   = nb.params["post"]
p_two = nb.pvalues["post"]
p_one = p_two/2 if b1>0 else 1-p_two/2     # 단측 p (H1: b1>0)
print(f"\n▶ β1(post) = {b1:.3f},  exp(β1) = {np.exp(b1):.2f}배,  단측 p = {p_one:.4g}")
print("  → 기사 수 유의 증가" if (b1>0 and p_one<0.05) else "  → 기사 수 증가 근거 약함")


## 7. 검정 ② 평균 유사도 — Permutation test

- **H₀:** 사후 평균 유사도 = 사전 &nbsp; **H₁:** 사후 > 사전 (단측)
- 유사도는 쌍끼리 독립이 아니므로 t검정 부적합 → **라벨 셔플 permutation**으로 영분포 생성.


In [ ]:
obs_diff = ts[ts.period=="post"]["S"].mean() - ts[ts.period=="pre"]["S"].mean()

labels = ts["period"].values.copy()
n_post = (labels=="post").sum()
S_vals = ts["S"].values

rng = np.random.default_rng(1)
perm = np.empty(N_PERM)
for i in range(N_PERM):
    sh = rng.permutation(S_vals)
    perm[i] = sh[:n_post].mean() - sh[n_post:].mean()

p_perm = (np.sum(perm >= obs_diff) + 1) / (N_PERM + 1)   # 단측
print(f"관측 차이 (사후-사전 평균유사도) = {obs_diff:.4f}")
print(f"Permutation 단측 p = {p_perm:.4g}")
print("  → 유사도 유의 증가" if p_perm<0.05 else "  → 유사도 증가 근거 약함")

plt.figure(figsize=(8,3.2))
plt.hist(perm, bins=50, color="#BFE6FF", edgecolor="white")
plt.axvline(obs_diff, color="#FF7A6B", lw=2.5, label=f"관측값 {obs_diff:.3f}")
plt.title("Permutation 영분포 (유사도 차이)"); plt.legend(); plt.tight_layout(); plt.show()


## 8. 검정 ②-보정 — 무작위 배정 영모형 (핵심 함정 제거)

**함정:** 기사 수가 늘면 평균 유사도가 *기계적으로* 오를 수 있다.
→ 각 사후 버킷의 관측 S_t를, **같은 개수의 기사를 전체 풀에서 무작위로 뽑았을 때**의 유사도 분포와 비교한다.

$$ z_t = \frac{S_t - \mu_{\text{rand}}(n_t)}{\sigma_{\text{rand}}(n_t)} $$

z가 크면 "기사가 많아서가 아니라 **진짜 서로 닮아서**"라는 증거.


In [ ]:
def null_similarity(n, n_sim, pool_emb, rng):
    """전체 풀에서 n개를 무작위로 뽑았을 때의 평균유사도 분포"""
    if n < 2:
        return np.nan, np.nan
    sims = np.empty(n_sim)
    N = len(pool_emb)
    for i in range(n_sim):
        sel = rng.choice(N, size=n, replace=False)
        sims[i] = mean_pairwise_cosine(pool_emb[sel])
    return sims.mean(), sims.std(ddof=1)

rng = np.random.default_rng(7)
rows = []
for _, r in ts.iterrows():
    n = int(r["N"])
    mu_r, sd_r = null_similarity(n, N_NULLSIM, emb, rng)
    z = (r["S"] - mu_r)/sd_r if (sd_r and sd_r>0) else np.nan
    rows.append({"bucket": r["bucket"], "period": r["period"],
                 "N": n, "S": r["S"], "S_null": mu_r, "z": z})
zt = pd.DataFrame(rows)
print(zt.to_string(index=False))

# 사후 z가 사전보다 체계적으로 큰지 (단측 Mann-Whitney)
from scipy.stats import mannwhitneyu
z_pre  = zt[zt.period=="pre"]["z"].dropna()
z_post = zt[zt.period=="post"]["z"].dropna()
u, p_z = mannwhitneyu(z_post, z_pre, alternative="greater")
print(f"\n사전 z 중앙값={z_pre.median():.2f}, 사후 z 중앙값={z_post.median():.2f}")
print(f"Mann-Whitney 단측 p = {p_z:.4g}")
print("  → 기사 수 효과를 걷어내고도 유사도 유의 증가" if p_z<0.05 else "  → 보정 후 증가 근거 약함")


In [ ]:
# z-점수 시각화
plt.figure(figsize=(11,3.6))
colors = ["#5BA8FF" if p=="pre" else "#FF7A6B" for p in zt.period]
plt.bar(range(len(zt)), zt["z"], color=colors)
plt.axhline(0, color="gray", lw=1)
plt.axhline(3, ls=":", color="#C23A2B", lw=2, label="z=3 (강한 신호)")
plt.axvline((zt.period=="pre").sum()-0.5, color="gray", lw=1)
plt.ylabel("z (관측 - 무작위기대)/표준편차")
plt.title("무작위 배정 영모형 대비 z-점수 (기사 수 효과 보정)")
plt.xticks(range(len(zt)),
           [b.strftime("%m-%d %Hh" if TIME_UNIT=="H" else "%m-%d") for b in zt.bucket],
           rotation=45, ha="right", fontsize=8)
plt.legend(); plt.tight_layout(); plt.show()


## 9. 종합 판정

세 검정을 합쳐 churnalism 여부를 판정합니다.

| 검정 | 내용 | 조건 |
|---|---|---|
| ① 기사 수 | 음이항 β₁>0 | 단측 p<0.05 |
| ② 유사도 | Permutation | 단측 p<0.05 |
| ②-보정 | 무작위 영모형 z | Mann-Whitney p<0.05 |

**①과 (②·②보정)이 모두 유의**하면 "기사 밀어내기(유사 기사 대량 생산)"로 판정.


In [ ]:
cond_count = (b1>0 and p_one<0.05)
cond_sim   = (p_perm<0.05)
cond_sim_adj = (p_z<0.05)

print("="*52)
print("  churnalism 검정 결과 요약")
print("="*52)
print(f"  ① 기사 수 (음이항 β1)      : p={p_one:.4g}  {'✅' if cond_count else '❌'}  (×{np.exp(b1):.2f})")
print(f"  ② 유사도 (permutation)     : p={p_perm:.4g}  {'✅' if cond_sim else '❌'}")
print(f"  ②-보정 (무작위 영모형 z)   : p={p_z:.4g}  {'✅' if cond_sim_adj else '❌'}")
print("-"*52)
verdict = cond_count and cond_sim and cond_sim_adj
print(f"  ▶ 최종 판정: {'🚨 기사 밀어내기 확실 (대량 생산 + 진짜 유사)' if verdict else '판정 보류 — 조건 미충족'}")
print("="*52)


---
### 다음 단계 / 커스터마이즈
- **실제 데이터**: 2번 셀에서 `pd.read_csv(...)`로 교체 (`published_at`, `text` 필수).
- **시간 해상도**: `TIME_UNIT='H'` 로 바꾸면 시간 단위 + 시간대 더미 자동 적용.
- **여러 이슈 비교**: `EVENT_TIME`을 바꿔가며 반복하거나, 이슈별로 루프.
- **임베딩 교체**: `EMB_MODEL` 변경 (예: `snunlp/KR-SBERT-V40K-klueNLI-augSTS`).
- **그림책(개념 설명)**: https://sdkparkforbi.github.io/news-pushing-book/

> 본 노트북은 "유사 기사 대량 생산"을 (1)음이항 회귀로 **양(N)**, (2)permutation+영모형으로 **닮음(S)**, 두 축에서 각각 검정하고 둘 다 충족 시 churnalism으로 판정한다.
